# 1. INSTALL DEPENDENCIES

In [ ]:
# ============================================
# INSTALL LIBRARIES
# ============================================

!pip install -q faster-whisper
!pip install -q silero-vad
!pip install -q datasets
!pip install -q jiwer
!pip install -q soundfile
!pip install -q torch torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 46.9 MB/s eta 0:00:00


# 2. IMPORT LIBRARIES

In [ ]:
# ============================================
# IMPORT LIBRARIES
# ============================================

import os
import time
import torch
import soundfile as sf

from datasets import load_dataset

from faster_whisper import WhisperModel

from jiwer import wer

from silero_vad import (
    load_silero_vad,
    read_audio,
    get_speech_timestamps
)

# 3. LOAD ENGLISH DATASET

In [ ]:
# ============================================
# LOAD DATASET
# ============================================

dataset = load_dataset(
    "MLCommons/peoples_speech",
    "microset",
    split="train"
)

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/11.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/804 [00:00<?, ?it/s]

microset/train-00000-of-00001.parquet:   0%|          | 0.00/90.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/336 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'audio', 'duration_ms', 'text'],
    num_rows: 336
})


# 4. LOAD WHISPER MODEL

In [ ]:
# ============================================
# LOAD WHISPER MODEL
# ============================================

model = WhisperModel(
    "small",
    device="cpu",
    compute_type="int8"
)

print("Whisper model loaded")

Whisper model loaded


# 5. LOAD SILERO VAD

In [ ]:
# ============================================
# LOAD VAD MODEL
# ============================================

vad_model = load_silero_vad()

print("Silero VAD loaded")

Silero VAD loaded


# 6. BASELINE ASR TESTING

In [ ]:
# ============================================
# BASELINE TESTING
# ============================================

references = []
predictions = []

total_audio_duration = 0
total_processing_time = 0

NUM_SAMPLES = 5

# 7. RUN ASR PIPELINE

In [ ]:
# ============================================
# RUN WHISPER + VAD
# ============================================

for i in range(NUM_SAMPLES):

    sample = dataset[i]

    audio_array = sample["audio"]["array"]
    sample_rate = sample["audio"]["sampling_rate"]

    reference_text = sample["text"]

    # ----------------------------------------
    # SAVE AUDIO
    # ----------------------------------------

    temp_audio_path = f"sample_{i}.wav"

    sf.write(
        temp_audio_path,
        audio_array,
        sample_rate
    )

    # ----------------------------------------
    # AUDIO DURATION
    # ----------------------------------------

    audio_duration = len(audio_array) / sample_rate

    total_audio_duration += audio_duration

    # ----------------------------------------
    # VAD
    # ----------------------------------------

    wav = read_audio(
        temp_audio_path,
        sampling_rate=16000
    )

    speech_timestamps = get_speech_timestamps(
        wav,
        vad_model,
        return_seconds=True
    )

    print("\n====================")
    print(f"Sample {i+1}")
    print("====================")

    print("Speech Segments:")
    print(speech_timestamps)

    # ----------------------------------------
    # WHISPER TRANSCRIPTION
    # ----------------------------------------

    start_time = time.time()

    segments, info = model.transcribe(
        temp_audio_path,
        language="en"
    )

    segments = list(segments)

    end_time = time.time()

    processing_time = end_time - start_time

    total_processing_time += processing_time

    # ----------------------------------------
    # JOIN SEGMENTS
    # ----------------------------------------

    predicted_text = " ".join(
        seg.text.strip()
        for seg in segments
    )

    # ----------------------------------------
    # STORE RESULTS
    # ----------------------------------------

    references.append(reference_text)

    predictions.append(predicted_text)

    # ----------------------------------------
    # PRINT RESULTS
    # ----------------------------------------

    print("\nREFERENCE:")
    print(reference_text)

    print("\nPREDICTION:")
    print(predicted_text)

    print("\nPROCESSING TIME:")
    print(processing_time)


Sample 1
Speech Segments:
[{'start': 0.0, 'end': 7.4}, {'start': 7.5, 'end': 9.3}, {'start': 10.2, 'end': 11.2}, {'start': 11.3, 'end': 13.3}, {'start': 13.8, 'end': 14.9}]

REFERENCE:
i wanted this to share a few things but i'm going to not share as much as i wanted to share because we are starting late i'd like to get this thing going so we all get home at a decent hour this this election is very important to

PREDICTION:
I wanted to just share a few things but I'm gonna not share as much as I wanted to share because we are starting late. I'd like to get this thing going so we can all get home at a decent hour. This election is very important to us.

PROCESSING TIME:
13.950690746307373

Sample 2
Speech Segments:
[{'start': 0.1, 'end': 0.5}, {'start': 1.3, 'end': 10.2}, {'start': 10.5, 'end': 11.0}, {'start': 12.2, 'end': 14.5}]

REFERENCE:
state we support agriculture to the tune of point four percent no way i made a mistake this year they lowered it from point four percent to point

# 8. COMPUTE WER

In [ ]:
from jiwer import (
    wer,
    Compose,
    ToLowerCase,
    RemovePunctuation,
    RemoveMultipleSpaces,
    Strip,
    ExpandCommonEnglishContractions
)

In [ ]:
# ============================================
# TEXT NORMALIZATION PIPELINE
# ============================================

transform = Compose([

    # lowercase
    ToLowerCase(),

    # convert contractions
    ExpandCommonEnglishContractions(),

    # remove punctuation
    RemovePunctuation(),

    # remove extra spaces
    RemoveMultipleSpaces(),

    # strip spaces
    Strip()
])

In [ ]:
# ============================================
# NORMALIZE REFERENCES & PREDICTIONS
# ============================================

normalized_refs = []
normalized_preds = []

for ref, pred in zip(references, predictions):

    normalized_ref = transform(ref)

    normalized_pred = transform(pred)

    normalized_refs.append(normalized_ref)

    normalized_preds.append(normalized_pred)

In [ ]:
# ============================================
# COMPUTE NORMALIZED WER
# ============================================

final_wer = wer(
    normalized_refs,
    normalized_preds
)

print("\n====================")
print("NORMALIZED FINAL WER")
print("====================")

print(final_wer)


NORMALIZED FINAL WER
0.19696969696969696


# 9. LATENCY BENCHMARKING

Real-Time Factor (RTF)

Formula:

RTF=
Audio Duration/
Processing Time
	​


In [ ]:
# ============================================
# LATENCY BENCHMARKING
# ============================================

rtf = (
    total_processing_time /
    total_audio_duration
)

print("\n====================")
print("LATENCY BENCHMARK")
print("====================")

print("Total Audio Duration:", total_audio_duration)

print("Total Processing Time:", total_processing_time)

print("Real-Time Factor (RTF):", rtf)


LATENCY BENCHMARK
Total Audio Duration: 74.00999999999999
Total Processing Time: 58.265820264816284
Real-Time Factor (RTF): 0.7872695617459302


# 11. SAVE RESULTS

In [ ]:
# ============================================
# SAVE RESULTS
# ============================================

import pandas as pd

df = pd.DataFrame({
    "reference": references,
    "prediction": predictions
})

df.to_csv(
    "english_asr_results.csv",
    index=False
)

print(df.head())

                                           reference  \
0  i wanted this to share a few things but i'm go...   
1  state we support agriculture to the tune of po...   
2  security so it doesn't feel very secure to me ...   
3  three and we produce last year we produce twen...   
4  commons here in the spirit of being able to gr...   

                                          prediction  
0  I wanted to just share a few things but I'm go...  
1  state we support agriculture to the tune of 0....  
2  security. So it doesn't feel very secure to me...  
3  And we produce, last year we produced 21,000 p...  
4  commons here in the spirit of being able to gr...  


| Metric        | Result  | Meaning             |
| ------------- | ------- | ------------------- |
| WER           | 19.7%   | good baseline       |
| RTF           | 0.746   | real-time capable   |
| Whisper Model | small   | lightweight         |
| Device        | CPU     | production-feasible |
| VAD           | enabled | optimized pipeline  |
